# TEKNOFEST Healthcare AI — Phase 02: Specialized Model Research
## Missense Variant Pathogenicity Classification

**Objective**: Research specialized models, evaluate applicability, and define competition strategy.

This notebook provides an interactive companion to the Phase 02 research report.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
print("Libraries loaded.")


## A. Project Context Recap

In [ ]:
from pathlib import Path

DATA_DIR = Path("EĞİTİM (TRAIN) SETLERİ 2")
master = pd.read_csv(DATA_DIR / "YARISMA_TRAIN_MASTER.csv")

print(f"MASTER: {master.shape[0]} rows x {master.shape[1]} cols")
print(f"Label distribution: {dict(master['Label'].value_counts())}")
print(f"Positive ratio: {master['Label'].mean():.3f}")
print(f"Mean row missingness: {master.isnull().mean(axis=1).mean()*100:.1f}%")


## B. Feature Pattern Analysis

The AL features follow a clear triplet pattern:
1. **Continuous score** [0, ~0.9]: allele frequency from a specific population
2. **Binary flag** {0, 1}: whether data exists for this population
3. **Secondary flag**: filter status or related annotation


In [ ]:
# Demonstrate the triplet pattern
al_cols = [c for c in master.columns if c.startswith("AL_")]

print("Feature structure demonstration (AL_39 to AL_50):")
print("-" * 80)
for i in range(39, 51):
    col = f"AL_{i}"
    if col in master.columns:
        s = master[col].dropna()
        nu = s.nunique()
        is_binary = nu <= 3
        pattern = "BINARY FLAG" if is_binary else f"SCORE [0, {s.max():.3f}]"
        print(f"  {col:>8} | n={len(s):>5} | unique={nu:>5} | {pattern}")

print()
print("Interpretation:")
print("  AL_39: Population membership flag (few values: 0, 1, or special)")
print("  AL_40: Allele frequency score (continuous, unique per variant)")
print("  AL_41: Data availability flag (binary: 0 or 1)")
print("  AL_42: Filter/quality flag (binary: 0 or 1)")
print("  AL_43: Next allele frequency score...")
print("  ... Pattern repeats for each population in database")


## C. EK Feature Deep Dive (Pre-computed Scores)

The EK features are the most informative and the most risky.
They appear to be pre-computed from external models.


In [ ]:
ek_cols = [c for c in master.columns if c.startswith("EK_")]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for idx, col in enumerate(ek_cols):
    ax = axes[idx // 3][idx % 3]
    s = master[col].dropna()

    for label, color, name in [(0, '#2196F3', 'Benign'), (1, '#F44336', 'Pathogenic')]:
        subset = master[master['Label'] == label][col].dropna()
        if len(subset) > 10:
            subset.plot(kind='kde', ax=ax, color=color, label=name, alpha=0.7)

    ax.set_title(f"{col}\nrange=[{s.min():.2f}, {s.max():.2f}], miss={master[col].isnull().mean()*100:.1f}%")
    ax.legend(fontsize=8)

plt.suptitle("EK Feature Distributions by Class", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# EK feature identification table
ek_info = []
for col in ek_cols:
    s = master[col].dropna()
    corr = master[['Label', col]].dropna().corr().iloc[0, 1]
    ek_info.append({
        "Feature": col,
        "Range": f"[{s.min():.3f}, {s.max():.3f}]",
        "Mean": f"{s.mean():.3f}",
        "Missing%": f"{master[col].isnull().mean()*100:.1f}%",
        "Corr w/Label": f"{corr:.4f}",
        "Likely Identity": {
            "EK_1": "CADD phred (compressed) or gene constraint",
            "EK_2": "CADD raw or FATHMM",
            "EK_3": "Unknown deleteriousness score",
            "EK_4": "REVEL, BayesDel, or ClinPred [0,1]",
            "EK_5": "phastCons or MetaLR/MetaSVM",
            "EK_6": "REVEL, BayesDel, or AlphaMissense [0,1]",
            "EK_7": "phyloP (EXACT range match)",
            "EK_8": "Grantham distance (normalized) or SiPhy",
            "EK_9": "GERP++ RS (EXACT range match)",
        }.get(col, "Unknown"),
        "Circularity Risk": {
            "EK_1": "LOW", "EK_2": "LOW", "EK_3": "LOW-MEDIUM",
            "EK_4": "HIGH", "EK_5": "LOW-MEDIUM", "EK_6": "MEDIUM-HIGH",
            "EK_7": "NONE", "EK_8": "LOW", "EK_9": "NONE",
        }.get(col, "Unknown")
    })

ek_df = pd.DataFrame(ek_info)
display(ek_df)


## D. Specialized Model Catalogue

### Key Models for Missense Variant Pathogenicity

| Category | Models | Likely in Dataset | ClinVar-Trained | Circularity Risk |
|----------|--------|-------------------|-----------------|------------------|
| Classical | SIFT, PolyPhen-2 | Yes (AL features) | Partially | LOW-MEDIUM |
| Meta-predictors | REVEL, BayesDel, ClinPred | Yes (EK_4/EK_6) | YES | HIGH |
| Deleteriousness | CADD, DANN, Eigen | Yes (EK_1/EK_2) | NO | LOW |
| Deep Learning | AlphaMissense, EVE | Possibly (EK_6?) | NO | LOW |
| Conservation | phyloP, GERP++ | Yes (EK_7, EK_9) | NO | NONE |

### Critical Insight
Our competition model IS a meta-predictor. The dataset provides pre-computed outputs from
multiple specialized models, and we build a classifier on top of them.
This is architecturally identical to REVEL and BayesDel.


## E. Strategy Comparison

In [ ]:
strategies = pd.read_csv("reports/phase_02_specialized_model_research/strategy_comparison.csv")
display(strategies[['Strategy', 'Performance Potential', 'Leakage Risk', 'Final Recommendation']])


## F. Risk Register

In [ ]:
risks = pd.read_csv("reports/phase_02_specialized_model_research/risk_register.csv")
display(risks[['Risk ID', 'Risk', 'Severity', 'Probability']])


## G. Feature Overlap Visualization

In [ ]:
# Feature group architecture visualization
feature_groups = {
    'Population AF\n(gnomAD exome)': (0, 26, '#4CAF50'),
    'Rare pop AF': (27, 38, '#81C784'),
    'Population AF\n(gnomAD genome\n+ others)': (39, 95, '#66BB6A'),
    'Additional AF\nmetadata': (96, 98, '#A5D6A7'),
    'Population AF\n(AllofUs +\nothers)': (99, 185, '#43A047'),
    'Predictor\nscores /\ntransition zone': (186, 222, '#FFA726'),
    'In-silico\npredictor\nscores': (223, 334, '#FF7043'),
}

fig, ax = plt.subplots(figsize=(16, 3))
for label, (start, end, color) in feature_groups.items():
    ax.barh(0, end - start + 1, left=start, height=0.6, color=color, edgecolor='white', linewidth=0.5)
    mid = start + (end - start) / 2
    ax.text(mid, 0, label, ha='center', va='center', fontsize=7, fontweight='bold')

ax.set_xlim(0, 340)
ax.set_ylim(-0.5, 0.5)
ax.set_xlabel("AL Feature Index")
ax.set_title("Inferred Feature Architecture (AL_1 to AL_334)", fontsize=14, fontweight='bold')
ax.set_yticks([])
plt.tight_layout()
plt.show()

print("\nEK Feature Layer (separate from AL):")
print("  EK_1-EK_3: Deleteriousness scores (CADD-like)")
print("  EK_4-EK_6: Meta-predictor scores (REVEL/BayesDel-like)")
print("  EK_7: phyloP conservation score")
print("  EK_8: Unknown (possibly Grantham or SiPhy)")
print("  EK_9: GERP++ conservation score")


## H. Circularity Analysis

The key question: if labels come from ClinVar, and some features (EK_4, EK_6) are ClinVar-trained
meta-predictors, how much does performance inflate?


In [ ]:
# Demonstrate circularity risk by comparing EK vs non-EK feature importance
from scipy.stats import mannwhitneyu

path_idx = master['Label'] == 1
ben_idx = master['Label'] == 0

# Compare EK features vs best AL features
comparison = []
for col in ['EK_4', 'EK_6', 'EK_7', 'EK_9', 'AL_26', 'AL_287', 'AL_215', 'AL_73']:
    sp = master.loc[path_idx, col].dropna()
    sb = master.loc[ben_idx, col].dropna()
    if len(sp) > 5 and len(sb) > 5:
        u, p = mannwhitneyu(sp, sb, alternative='two-sided')
        r = 1 - (2 * u) / (len(sp) * len(sb))
        comparison.append({
            "Feature": col,
            "Effect Size |r|": round(abs(r), 4),
            "p-value": f"{p:.2e}",
            "Likely Type": "Meta-predictor" if col in ['EK_4', 'EK_6'] else
                          "Conservation" if col in ['EK_7', 'EK_9'] else "Allele Freq",
            "Circularity Risk": "HIGH" if col in ['EK_4', 'EK_6'] else
                               "NONE" if col in ['EK_7', 'EK_9'] else "LOW"
        })

comp_df = pd.DataFrame(comparison)
display(comp_df)

print("\nKey observation:")
print("If EK_4/EK_6 are much more predictive than EK_7/EK_9 and AF features,")
print("this suggests the model may be relying on ClinVar-circular features.")
print("We MUST train a model WITHOUT EK_4/EK_6 to measure honest performance.")


## I. Proposed Architecture

```
RAW FEATURES → Feature Engineering → [LightGBM, XGBoost, CatBoost] → Stacking → Calibration → Threshold → SHAP
```

### Phase 3 Implementation Plan:
1. LightGBM baseline on all features
2. Feature engineering (missingness indicators, AA encoding)
3. XGBoost + CatBoost models
4. Stacking ensemble
5. Panel-specific calibration
6. EK-excluded circularity analysis
7. SHAP explainability
8. Clinical thresholding (sensitivity ≥ 95%)
9. Final report

### B-Plan:
- If ensemble fails: single LightGBM
- If EK features circular: drop EK_4/EK_5/EK_6
- If CFTR too small: global model + CFTR threshold only
- If too much missingness: features with <50% missing only


## J. Final Recommendation

**PRIMARY STRATEGY**: Custom GBDT ensemble (LightGBM + XGBoost + CatBoost) framed as a
competition-specific meta-predictor, with panel calibration and sensitivity-optimized thresholding.

**CRITICAL ACTION**: Train models WITH and WITHOUT EK_4/EK_6 to quantify circularity impact.

**REPORT FRAMING**: Present as a meta-predictor that combines population frequencies, conservation
scores, and existing predictor outputs — the same architecture as REVEL/BayesDel, but competition-specific.
